<a href="https://colab.research.google.com/github/srijitaaa2005/MY-WORKS-DEEP-LEARNING-/blob/main/cats_vs_dogs_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"srijitaaa18","key":"f0e2c6fabd2e5305a40b2dca5b3b84e7"}'}

In [2]:
!pip install -q kaggle

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [4]:
!kaggle datasets download -d pushpakhinglaspure/cats-vs-dogs

Dataset URL: https://www.kaggle.com/datasets/pushpakhinglaspure/cats-vs-dogs
License(s): DbCL-1.0
100% 546M/546M [00:05<00:00, 107MB/s]



In [5]:
import zipfile
zip_ref = zipfile.ZipFile('/content/cats-vs-dogs.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [6]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, BatchNormalization, Dropout

In [12]:
# #generators
# train_ds=keras.utils.image_dataset_from_directory(
#     directory='/content/dogs_vs_cats/dogs_vs_cats/train',
#     labels='inferred',
#     label_mode='int', #cats=0, dogs=1
#     batch_size=32,
#     image_size=(256,256)
# )
# validation_ds=keras.utils.image_dataset_from_directory(
#     directory='/content/dogs_vs_cats/dogs_vs_cats/test',
#     labels='inferred',
#     label_mode='int', #cats=0, dogs=1
#     batch_size=32,
#     image_size=(256,256)
# )

In [13]:
# #normalize
# def process(image,label):
#   image=tf.cast(image/255, tf.float32)
#   return image, label
# train_ds=train_ds.map(process)
# validation_ds=validation_ds.map(process)

**DATA AUGMENTATION**

In [14]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

batch_size=16

train_datagen=ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)
test_datagen=ImageDataGenerator(rescale=1./256)

train_generator=train_datagen.flow_from_directory(
    '/content/dogs_vs_cats/dogs_vs_cats/train',
    target_size=(256,256),
    batch_size= batch_size,
    class_mode='binary'
)
validation_generator=test_datagen.flow_from_directory(
    '/content/dogs_vs_cats/dogs_vs_cats/test',
    target_size=(256,256),
    batch_size= batch_size,
    class_mode='binary'
)

Found 20000 images belonging to 2 classes.
Found 5000 images belonging to 2 classes.


In [15]:
#cnn architechture
model=Sequential()

model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=(2,2),padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=(2,2),padding='valid'))

model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=(2,2),padding='valid'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1,activation='sigmoid'))

model.summary()



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 254, 254, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 125, 125, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 60, 60, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,193 (56.64 MB)

 Trainable params: 14,847,745 (56.64 MB)

 Non-trainable params: 448 (1.75 KB)

In [17]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
#history = model.fit(train_ds, epochs=10, validation_data=validation_ds)

history=model.fit(
    train_generator,
    steps_per_epoch=2000 // batch_size,
    epochs=30,
    validation_data=validation_generator,
    validation_steps=800 // batch_size)

Epoch 1/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 40s 273ms/step - accuracy: 0.6795 - loss: 0.6386 - val_accuracy: 0.5412 - val_loss: 1.2477
Epoch 2/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 33s 268ms/step - accuracy: 0.6635 - loss: 0.6625 - val_accuracy: 0.7050 - val_loss: 0.6227
Epoch 3/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 32s 258ms/step - accuracy: 0.6620 - loss: 0.6437 - val_accuracy: 0.7075 - val_loss: 0.5766
Epoch 4/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 32s 262ms/step - accuracy: 0.6680 - loss: 0.6426 - val_accuracy: 0.5962 - val_loss: 0.6391
Epoch 5/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 31s 253ms/step - accuracy: 0.7020 - loss: 0.6119 - val_accuracy: 0.5587 - val_loss: 0.8683
Epoch 6/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 33s 263ms/step - accuracy: 0.6955 - loss: 0.5982 - val_accuracy: 0.5850 - val_loss: 0.6881
Epoch 7/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 32s 261ms/step - accuracy: 0.7140 - loss: 0.5794 - val_accuracy: 0.6525 - val_loss: 0.7110
Epoch 8/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 32s 257ms/step - accuracy: 0.7300 - loss: 0

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'],color='red',label='train')
plt.plot(history.history['val_accuracy'],color='blue',label='validation')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'],color='red',label='train')
plt.plot(history.history['val_loss'],color='blue',label='validation')
plt.legend()
plt.show()

In [ ]:
print("Training Accuracy:", history.history['accuracy'][-1])
print("Validation Accuracy:", history.history['val_accuracy'][-1])

In [ ]:
print(train_generator.class_indices)

In [ ]:
import cv2

test_img_dog=cv2.imread('/content/dog.jpg')
plt.imshow(test_img_dog)

In [ ]:
test_img_dog=cv2.resize(test_img_dog,(256,256))
test_input_dog=test_img_dog.reshape((1,256,256,3))
model.predict(test_input_dog)

In [ ]:
test_img_cat=cv2.imread('/content/cat.jpg')
plt.imshow(test_img_cat)

In [ ]:
test_img_cat=cv2.resize(test_img_cat,(256,256))
test_input_cat=test_img_cat.reshape((1,256,256,3))
model.predict(test_input_cat)